# Run the toolkit on 1,000 landing pages

*Python Foundations · Topic 12.4*

The marketing team wants the store's landing page report with every link's UTMs cleaned, its campaign name checked and its channel group, as a file they can open in a spreadsheet. They also want to know how many sessions each channel group brings, and which campaign names break the naming convention.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Download [landing_pages.csv](https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/landing_pages.csv) and upload it to the `simmer-python` folder in your Drive. Run the parameters cell and the cell that mounts Drive.
2. In the first cell marked ✏️, paste your toolkit from topic 12.3.: everything in the editor from `from urllib.parse import ...` to the end of `channel_group()`, without the real values below it.
3. In the second cell marked ✏️, create `clean_rows`: a list with one dictionary per row of `rows`, with these keys, in this order:
   - `landing_page`: the row's landing page
   - `utm_source`, `utm_medium`, `utm_campaign` and `utm_content`: from `parse_utm()`
   - `campaign_valid`: `is_valid_campaign()` of the campaign
   - `channel_group`: `channel_group()` of the source and medium
   - `sessions`: the row's sessions, as an integer
4. In the third cell marked ✏️, write `clean_rows` to `OUTPUT_PATH` with `csv.DictWriter`, with the columns in the same order.
5. In the last cell marked ✏️, create:
   - `sessions_by_channel`: a dictionary with each channel group's total sessions
   - `invalid_campaigns`: a sorted list of the campaign names in `clean_rows` that aren't valid, each one once, without `None`
6. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output. Then open `landing_pages_clean.csv` in your Drive with Google Sheets.

**Expected output**: `clean_rows` has 1,000 rows, the file has a header and 1,000 rows, `invalid_campaigns` has 43 names, and `sessions_by_channel` is `{'Paid Search': 54272, 'Paid Social': 45130, 'Paid Video': 33695, 'Email': 29561, 'Unassigned': 21451, 'Display': 19288}`. The order of the keys doesn't matter.

## Parameters

In [ ]:
CSV_PATH = "/content/drive/MyDrive/simmer-python/landing_pages.csv"
OUTPUT_PATH = "/content/drive/MyDrive/simmer-python/landing_pages_clean.csv"

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## Toolkit

In [ ]:
# ✏️ Paste your toolkit from topic 12.3: from the import to the end of channel_group()


## Landing pages

The store's landing page report: one row per landing page URL, with its sessions.

In [ ]:
import csv

with open(CSV_PATH, newline="", encoding="utf-8") as file:
    rows = list(csv.DictReader(file))
len(rows)

In [ ]:
# ✏️ Create clean_rows: one dictionary per row, with the keys in the brief's order


## Export

In [ ]:
# ✏️ Write clean_rows to OUTPUT_PATH with csv.DictWriter


## Summary

In [ ]:
# ✏️ Create sessions_by_channel and invalid_campaigns


In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables, the file you wrote and the cells that have run. It doesn't change anything.
def check_your_work():
    import csv
    import os
    import sys
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    def call(name, *args):
        fn = found.get(name)
        try:
            return fn(*args)
        except Exception as error:
            return f"{type(error).__name__}: {error}"

    missing = [name for name in ["parse_utm", "is_valid_campaign", "channel_group"] if not callable(found.get(name))]
    if missing:
        results.append((False, f"{', '.join(missing)} isn't defined. Paste your toolkit from topic 12.3 into the first ✏️ cell: everything from the import to the end of channel_group()."))
    else:
        cases = [
            ("parse_utm", ("https://store.teamsimmer.com/c/t-shirts?utm_source=Facebook&UTM_CAMPAIGN=2024_us_facebook_prospecting#top",), {"utm_source": "facebook", "utm_medium": None, "utm_campaign": "2024_us_facebook_prospecting", "utm_content": None}),
            ("parse_utm", ("https://store.teamsimmer.com/c/outerwear",), {"utm_source": None, "utm_medium": None, "utm_campaign": None, "utm_content": None}),
            ("is_valid_campaign", ("2025_ca_email_springsale",), True),
            ("is_valid_campaign", ("2025_ca_youtube_awareness_v2",), False),
            ("is_valid_campaign", (None,), False),
            ("channel_group", ("lm.facebook.com", "referral"), "Organic Social"),
            ("channel_group", ("facebook", None), "Unassigned"),
            ("channel_group", ("Youtube", "VIDEO"), "Paid Video"),
        ]
        wrong = [(name, args, expected, call(name, *args)) for name, args, expected in cases if call(name, *args) != expected]
        if wrong:
            name, args, expected, got = wrong[0]
            shown = ", ".join(repr(arg) for arg in args)
            results.append((False, f"Your toolkit isn't the one from topic 12.3: {name}({shown}) should return {expected!r}, but it returned {got!r}. Copy the whole toolkit from the editor after its checks pass, and paste it again."))
        else:
            results.append((True, "The toolkit is the one from topic 12.3."))

    rows = found.get("rows")
    path = found.get("CSV_PATH")
    if not isinstance(path, str) or not os.path.exists(path):
        results.append((False, f"There's no file at CSV_PATH ({path!r}). Upload landing_pages.csv to the simmer-python folder in your Drive, and run the cell that mounts Drive."))
    elif not isinstance(rows, list) or len(rows) != 1000:
        results.append((False, "rows should have the file's 1,000 rows. Run the cell that reads it."))
    else:
        results.append((True, "rows has all 1,000 landing pages."))

    keys = ["landing_page", "utm_source", "utm_medium", "utm_campaign", "utm_content", "campaign_valid", "channel_group", "sessions"]
    clean_rows = found.get("clean_rows")
    expected_second = {"landing_page": "https://store.teamsimmer.com/c/t-shirts?utm_source=facebook&utm_campaign=2024_us_facebook_prospecting", "utm_source": "facebook", "utm_medium": None, "utm_campaign": "2024_us_facebook_prospecting", "utm_content": None, "campaign_valid": True, "channel_group": "Unassigned", "sessions": 293}
    rows_ok = False
    if not isinstance(clean_rows, list) or not all(isinstance(row, dict) for row in clean_rows):
        results.append((False, "clean_rows should be a list of dictionaries, one per row of rows."))
    elif len(clean_rows) != 1000:
        results.append((False, f"clean_rows has {len(clean_rows)} rows, but rows has 1,000. Add one dictionary for every row, not only a sample."))
    elif list(clean_rows[1]) != keys:
        results.append((False, f"clean_rows' dictionaries have the keys {list(clean_rows[1])}. They should have these keys, in this order: {keys}."))
    elif isinstance(clean_rows[1]["sessions"], str):
        results.append((False, "sessions is text in clean_rows. Convert it with int(row[\"sessions\"])."))
    elif clean_rows[1] != expected_second:
        results.append((False, f"clean_rows[1] should be\n   {expected_second}\n   but it is\n   {clean_rows[1]}"))
    elif sum(row["sessions"] for row in clean_rows) != 203397 or sum(row["campaign_valid"] is True for row in clean_rows) != 897:
        results.append((False, "clean_rows has the right keys, but not the right values in every row. Check that each row uses its own landing page and the toolkit's results for it."))
    else:
        rows_ok = True
        results.append((True, "clean_rows has the toolkit's results for all 1,000 landing pages."))

    output = found.get("OUTPUT_PATH")
    if not isinstance(output, str) or not os.path.exists(output):
        results.append((False, f"There's no file at OUTPUT_PATH ({output!r}). Write clean_rows to it with csv.DictWriter, in the third ✏️ cell."))
    else:
        with open(output, newline="", encoding="utf-8") as file:
            reader = csv.DictReader(file)
            header = reader.fieldnames
            saved = list(reader)
        if header != keys:
            results.append((False, f"The file's columns are {header}. Write it with fieldnames in the brief's order: {keys}."))
        elif len(saved) != 1000:
            results.append((False, f"The file has {len(saved)} rows, but clean_rows has 1,000. Write all of them with writer.writerows(clean_rows)."))
        elif saved[1] != {"landing_page": expected_second["landing_page"], "utm_source": "facebook", "utm_medium": "", "utm_campaign": "2024_us_facebook_prospecting", "utm_content": "", "campaign_valid": "True", "channel_group": "Unassigned", "sessions": "293"}:
            results.append((False, f"The file's second row is {saved[1]}. Write clean_rows as it is, with csv.DictWriter: None becomes an empty cell."))
        else:
            results.append((True, "The file has a header and all 1,000 rows, with the columns in order."))

    expected_channels = {"Paid Search": 54272, "Paid Social": 45130, "Paid Video": 33695, "Email": 29561, "Unassigned": 21451, "Display": 19288}
    totals = found.get("sessions_by_channel")
    if totals == expected_channels:
        results.append((True, "sessions_by_channel has each channel group's sessions."))
    elif isinstance(totals, dict) and sum(v for v in totals.values() if isinstance(v, int)) == 1000:
        results.append((False, "sessions_by_channel counts rows, not sessions. Add row[\"sessions\"] to each channel group's total."))
    elif isinstance(totals, dict) and rows_ok:
        results.append((False, f"sessions_by_channel should be {expected_channels}, but it is {totals}. Total row[\"sessions\"] by row[\"channel_group\"] over clean_rows, with .get(group, 0)."))
    else:
        results.append((False, "sessions_by_channel should be a dictionary with each channel group's total sessions in clean_rows."))

    expected_invalid = ["2024-ca-display-prospecting", "2024-us-display-retargeting", "2024-us-search-shopping", "2024_us_facebook_prospecting_v2", "2024_us_youtube", "2025-ca-facebook-retargeting", "2025-ca-youtube-awareness", "2025-us-instagram-prospecting", "2025-us-search-generic", "2025_ca_display", "2025_ca_display_retargeting_v2", "2025_ca_facebook", "2025_ca_search_brand_v2", "2025_ca_youtube_awareness_v2", "2025_us_email_newsletter_v2", "2025_us_search", "2025_us_search_shopping_v2", "2025_us_youtube", "2025_us_youtube_awareness_v2", "24_ca_facebook_prospecting", "24_ca_search_brand", "24_ca_youtube_awareness", "25_ca_instagram_prospecting", "25_us_email_blackfriday", "25_us_facebook_holiday", "25_us_facebook_prospecting", "25_us_instagram_prospecting", "25_us_search_shopping", "awareness2024", "awareness2025", "ca_facebook_prospecting_2025", "ca_youtube_awareness_2024", "ca_youtube_awareness_2025", "holiday2024", "newsletter2024", "newsletter2025", "retargeting2024", "shopping2025", "us_email_newsletter_2024", "us_facebook_blackfriday_2025", "us_facebook_retargeting_2025", "us_search_generic_2025", "us_search_shopping_2025"]
    invalid = found.get("invalid_campaigns")
    if invalid == expected_invalid:
        results.append((True, "invalid_campaigns has the 43 names that break the naming convention."))
    elif not isinstance(invalid, list):
        results.append((False, "invalid_campaigns should be a sorted list: sorted({row[\"utm_campaign\"] for row in clean_rows if ...}). If sorted() stopped with TypeError: '<' not supported between instances of 'NoneType' and 'str', the set has None in it: leave out the rows without a campaign with row[\"utm_campaign\"] is not None."))
    elif None in invalid:
        results.append((False, "invalid_campaigns has None in it: the landing pages without a campaign. Leave them out with row[\"utm_campaign\"] is not None."))
    elif len(invalid) == 51:
        results.append((False, "invalid_campaigns has 51 names, with some of them twice. Put them in a set first, so each name is there once, then sort it."))
    elif sorted(invalid) == expected_invalid:
        results.append((False, "invalid_campaigns has the right names, but it isn't sorted. Use sorted()."))
    else:
        results.append((False, f"invalid_campaigns should have 43 names, but it has {len(invalid)}. Keep the campaigns whose campaign_valid is False and that aren't None."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run. Open landing_pages_clean.csv in your Drive: that's the file for the team.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
from urllib.parse import parse_qs, urlsplit

UTM_KEYS = ("utm_source", "utm_medium", "utm_campaign", "utm_content")
MARKETS = {"us", "ca"}
CAMPAIGN_CHANNELS = {"search", "display", "facebook", "instagram", "youtube", "email"}
CHANNEL_GROUPS = {
    "Direct", "Organic Search", "Paid Search", "Display", "Paid Social",
    "Paid Video", "Organic Social", "Referral", "Email", "Unassigned",
}
SOCIAL_SITES = {"facebook.com", "m.facebook.com", "l.facebook.com", "lm.facebook.com", "instagram.com", "l.instagram.com"}


def parse_utm(url: str) -> dict[str, str | None]:
    """Return the URL's four UTM parameters, cleaned.

    Args:
        url: A landing page URL.

    Returns:
        A dictionary with utm_source, utm_medium, utm_campaign and
        utm_content, in that order. Names match whatever their case, the
        first value wins, values are decoded, stripped and lowercased,
        and a parameter that's missing is None.
    """
    params = {key: None for key in UTM_KEYS}
    for name, values in parse_qs(urlsplit(url).query).items():
        name = name.lower()
        if name in params and params[name] is None:
            params[name] = values[0].strip().lower()
    return params


def is_valid_campaign(name: str | None) -> bool:
    """Return True if the campaign name follows the naming convention.

    Args:
        name: A campaign name, or None if the campaign is missing.

    Returns:
        True for {yyyy}_{market}_{channel}_{theme}: a four-digit year, a
        market in MARKETS, a channel in CAMPAIGN_CHANNELS and a theme of
        lowercase letters. False for any other name, and for None.
    """
    if name is None:
        return False
    parts = name.split("_")
    if len(parts) != 4:
        return False
    year, market, channel, theme = parts
    return (
        len(year) == 4 and year.isdigit()
        and market in MARKETS
        and channel in CAMPAIGN_CHANNELS
        and theme.isalpha() and theme.islower()
    )


def clean(value: str | None) -> str:
    """Return the value lowercased and stripped, or "" for None."""
    if value is None:
        return ""
    return value.strip().lower()


def channel_group(source: str | None, medium: str | None) -> str:
    """Return the channel group of a session's source and medium.

    Args:
        source: The session's source, such as "google", or None.
        medium: The session's medium, such as "cpc", or None.

    Returns:
        One of CHANNEL_GROUPS, with the CRM's names. The values are
        compared lowercased and stripped, and the spec's first matching
        rule wins. "Unassigned" if no rule matches, such as when the
        medium is None.
    """
    source = clean(source)
    medium = clean(medium)
    if source == "(direct)" and medium == "(none)":
        return "Direct"
    if medium == "email":
        return "Email"
    if medium == "cpc":
        return "Paid Search"
    if medium == "display":
        return "Display"
    if medium == "paid_social":
        return "Paid Social"
    if medium == "video":
        return "Paid Video"
    if medium == "organic":
        return "Organic Search"
    if medium == "referral" and source in SOCIAL_SITES:
        return "Organic Social"
    if medium == "referral":
        return "Referral"
    return "Unassigned"
```

```python
clean_rows = []
for row in rows:
    utm = parse_utm(row["landing_page"])
    clean_rows.append({
        "landing_page": row["landing_page"],
        "utm_source": utm["utm_source"],
        "utm_medium": utm["utm_medium"],
        "utm_campaign": utm["utm_campaign"],
        "utm_content": utm["utm_content"],
        "campaign_valid": is_valid_campaign(utm["utm_campaign"]),
        "channel_group": channel_group(utm["utm_source"], utm["utm_medium"]),
        "sessions": int(row["sessions"]),
    })
len(clean_rows)
```

```python
FIELDNAMES = ["landing_page", "utm_source", "utm_medium", "utm_campaign", "utm_content", "campaign_valid", "channel_group", "sessions"]

with open(OUTPUT_PATH, "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=FIELDNAMES)
    writer.writeheader()
    writer.writerows(clean_rows)
```

```python
sessions_by_channel = {}
for row in clean_rows:
    group = row["channel_group"]
    sessions_by_channel[group] = sessions_by_channel.get(group, 0) + row["sessions"]

invalid_campaigns = sorted({
    row["utm_campaign"] for row in clean_rows
    if row["utm_campaign"] is not None and not row["campaign_valid"]
})
sessions_by_channel, len(invalid_campaigns)
```

</details>